<a href="https://colab.research.google.com/github/dfmelgarc/consolibank/blob/main/app_compras_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

st.set_page_config(page_title="Proyección de Compras", layout="wide")

st.title("📦 Plan Maestro de Compras Predictivo 2024")
st.write("Sube tu histórico de ventas para generar los índices estacionales y la proyección de compras.")

# Sustituimos el rastreo automático de glob por un cargador de archivos visual
archivo_detectado = st.file_uploader("Sube tu archivo Excel (.xlsx)", type=["xlsx"])

if archivo_detectado is not None:
    with st.spinner("Procesando matriz de datos e identificando estacionalidad..."):
        # 1. Carga e Ingeniería de Datos (Tu código original adaptado)
        df_sistema = pd.read_excel(archivo_detectado)
        df_sistema['Fecha'] = pd.to_datetime(df_sistema['Fecha'])
        df_categoria_mes = df_sistema.groupby([pd.Grouper(key='Fecha', freq='ME'), 'Categoria'])['Cantidad'].sum().reset_index() #[cite: 1]
        df_series_categorias = df_categoria_mes.pivot(index='Fecha', columns='Categoria', values='Cantidad').fillna(1) #[cite: 1]

        st.success(f"Matriz estructurada con éxito. Líneas de negocio detectadas: {', '.join(list(df_series_categorias.columns))}")

        # 2. Análisis Estacional
        st.subheader("Análisis de Variación Estacional por Categoría")
        fig_estacional, ax_est = plt.subplots(figsize=(12, 5))

        for cat in df_series_categorias.columns:
            decomposicion = seasonal_decompose(df_series_categorias[cat], model='multiplicative', period=12) #[cite: 1]
            factores_mes = decomposicion.seasonal.groupby(decomposicion.seasonal.index.month).first() #[cite: 1]
            ax_est.plot(factores_mes.index, factores_mes.values, marker='o', linewidth=2, label=f'{cat}')

        ax_est.set_title('Factor Multiplicador de Demanda (Ene-Dic)')
        ax_est.set_xticks(range(1, 13))
        ax_est.set_xticklabels(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
        ax_est.axhline(1.0, color='red', linestyle='--', alpha=0.6) #[cite: 1]
        ax_est.grid(True, linestyle='--', alpha=0.5)
        ax_est.legend()
        st.pyplot(fig_estacional)

        # 3. Modelación Holt-Winters y Proyección
        st.subheader("Plan Maestro Sugerido (Siguientes 12 meses)")
        fechas_2024 = pd.date_range(start=df_series_categorias.index[-1] + pd.Timedelta(days=1), periods=12, freq='ME') #[cite: 1]
        plan_maestro_2024 = pd.DataFrame({'Mes_Proyectado': fechas_2024.strftime('%Y-%m')})

        for cat in df_series_categorias.columns:
            modelo_cat = ExponentialSmoothing(
                df_series_categorias[cat], trend='add', seasonal='mul', seasonal_periods=12 #[cite: 1]
            ).fit()
            prediccion = modelo_cat.forecast(12) #[cite: 1]
            plan_maestro_2024[f'Sugerido_{cat}'] = np.round(prediccion.values).astype(int) #[cite: 1]

        st.dataframe(plan_maestro_2024, use_container_width=True)

        # Botón de Descarga Profesional
        csv_buffer = plan_maestro_2024.to_csv(index=False).encode('utf-8')
        st.download_button(
            label="📥 Descargar Plan de Compras (CSV)",
            data=csv_buffer,
            file_name="Plan_Maestro_Compras.csv",
            mime="text/csv"
        )

ModuleNotFoundError: No module named 'streamlit'